# PaliGemma-3B LoRA — Multi-Turn Visual QA Demo

**Course:** Deep Learning with Computer Vision (DLCV)  
**Demo:** Fine-tuned PaliGemma-3B with LoRA adapters on LLaVA-Instruct-150K  

### Before running:
1. **Settings → Accelerator → GPU T4 x1**
2. **Add your training output as input:**  
   Notebook → Add Input → Your Work → select your training notebook output  
   *(adapters will appear at `/kaggle/input/.../dlcv_paligemma/lora_adapters/`)*
3. Add **HF_TOKEN** secret (Add-ons → Secrets)

### What this demo shows:
- Loading fine-tuned PaliGemma-3B with LoRA adapters
- Multi-turn visual conversation on 4 different scenes
- **Before vs After** fine-tuning comparison
- Ablation: effect of conversation history length

## Step 0 — GPU Check

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
assert torch.cuda.is_available(), 'No GPU — Settings → Accelerator → GPU T4'
assert torch.cuda.device_count() == 1, f'Expected 1 GPU, got {torch.cuda.device_count()}'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA : {torch.version.cuda}')
print('✅ GPU ready')

## Step 1 — Install Packages

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.44.0', 'peft==0.12.0', 'accelerate==0.33.0',
    'Pillow', 'requests', 'matplotlib', 'tqdm'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', '--no-deps', 'numpy==2.0.2'], check=True)

print('✅ Packages installed')

## Step 2 — Imports & Find Adapters

In [ ]:
import sys, os, glob, requests
from io import BytesIO
from unittest.mock import MagicMock

os.environ['TRANSFORMERS_NO_TF']   = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
sys.modules['bitsandbytes'] = None

try:
    from triton.ops.matmul_perf_model import early_config_prune
except (ImportError, ModuleNotFoundError):
    _mock = MagicMock()
    sys.modules['triton.ops']                   = _mock
    sys.modules['triton.ops.matmul_perf_model'] = _mock

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from IPython.display import display, HTML
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from huggingface_hub import login

# ── Find adapter directory ────────────────────────────────────────────────────
ADAPTER_DIR = None

# Search Kaggle input datasets first (added from previous training run)
candidates = glob.glob('/kaggle/input/**/lora_adapters', recursive=True)
if candidates:
    ADAPTER_DIR = candidates[0]
    print(f'Found adapters in input: {ADAPTER_DIR}')

# Fall back to current working directory (if training was just run)
if ADAPTER_DIR is None:
    fallback = '/kaggle/working/dlcv_paligemma/lora_adapters'
    if os.path.exists(fallback):
        ADAPTER_DIR = fallback
        print(f'Found adapters in working dir: {ADAPTER_DIR}')

if ADAPTER_DIR is None:
    raise FileNotFoundError(
        'Adapter directory not found!\n'
        'Add your training notebook output as input:\n'
        'Notebook → Add Input → Your Work → select training notebook'
    )

print(f'numpy {np.__version__}')
print(f'torch {torch.__version__}')
print('✅ Imports OK')

## Step 3 — HuggingFace Login

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('Loaded HF_TOKEN from Kaggle Secrets')
except Exception:
    HF_TOKEN = ''  # paste your token here if Secrets not set

assert HF_TOKEN, 'No HF token — add HF_TOKEN in Add-ons → Secrets'
login(token=HF_TOKEN)
print('✅ Logged in to HuggingFace')

## Step 4 — Load Fine-tuned Model

Loads PaliGemma-3B base + our LoRA adapters (44 MB of trained weights).

In [ ]:
MODEL_ID = 'google/paligemma-3b-pt-224'
MAX_LEN  = 192

print('Loading processor...')
processor = PaliGemmaProcessor.from_pretrained(ADAPTER_DIR, token=HF_TOKEN)

print('Loading base model (float16)...')
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={'': 0},
    token=HF_TOKEN,
)
base_model.config.use_cache = False

print('Applying LoRA adapters...')
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

vram  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\nVRAM : {vram:.1f} GB / {total:.1f} GB ({total-vram:.1f} GB free)')
print(f'Adapter: {ADAPTER_DIR}')

# Count parameters
total_p   = sum(p.numel() for p in model.parameters())
trained_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params   : {total_p/1e9:.2f}B')
print(f'Trainable (LoRA): {trained_p/1e6:.1f}M ({100*trained_p/total_p:.2f}%)')
print('✅ Model ready')

## Helper Functions

In [ ]:
def fetch_image(url):
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return Image.open(BytesIO(r.content)).convert('RGB')
    except Exception:
        return Image.new('RGB', (224, 224), (100, 100, 100))


def generate(image, prompt, max_new=60):
    inputs = processor(
        images=image, text=prompt,
        return_tensors='pt', padding='longest',
        truncation=True, max_length=MAX_LEN,
    ).to('cuda')
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, num_beams=3,
        )
    return processor.decode(out[0][input_len:], skip_special_tokens=True).strip()


def multiturn(image, questions):
    """Run multi-turn conversation and return list of (Q, A) pairs."""
    history = []
    for q in questions:
        parts = [f'Q: {h["q"]} A: {h["a"]}' for h in history]
        parts.append(f'Q: {q} A:')
        answer = generate(image, ' '.join(parts))
        history.append({'q': q, 'a': answer})
    return history


def show_demo(image, history, title=''):
    """Display image + conversation side by side."""
    fig, (ax_img, ax_txt) = plt.subplots(1, 2, figsize=(13, 4),
                                          gridspec_kw={'width_ratios': [1, 2]})
    ax_img.imshow(image)
    ax_img.axis('off')
    ax_img.set_title(title, fontsize=12, fontweight='bold', pad=8)

    ax_txt.axis('off')
    y = 0.97
    for i, turn in enumerate(history):
        q_text = f'Q{i+1}: {turn["q"]}'
        a_text = f'A{i+1}: {turn["a"]}'
        ax_txt.text(0, y, q_text, transform=ax_txt.transAxes,
                    fontsize=10, color='#3730a3', fontweight='bold',
                    wrap=True, va='top')
        y -= 0.07
        ax_txt.text(0, y, a_text, transform=ax_txt.transAxes,
                    fontsize=10, color='#374151', wrap=True, va='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='#eef2ff', alpha=0.7))
        y -= 0.18
        if y < 0.05:
            break

    plt.tight_layout()
    plt.savefig(f'/kaggle/working/demo_{title.replace(" ","_").lower()}.png',
                dpi=130, bbox_inches='tight')
    plt.show()
    print()


print('✅ Helper functions ready')

## Demo 1 — Food Scene (Multi-Turn)

In [ ]:
img_food = fetch_image('http://images.cocodataset.org/train2017/000000000009.jpg')

questions_food = [
    'What food items are visible in this image?',
    'What colors are the containers?',
    'How many separate food sections do you see?',
    'Is this a healthy meal? Why?',
]

print('Running multi-turn conversation on food scene...')
history_food = multiturn(img_food, questions_food)
show_demo(img_food, history_food, title='Food Scene')

## Demo 2 — Animals (Multi-Turn)

In [ ]:
img_animal = fetch_image('http://images.cocodataset.org/train2017/000000000025.jpg')

questions_animal = [
    'What animal is in this image?',
    'What is the animal doing?',
    'Describe the environment around the animal.',
    'What time of day does this appear to be?',
]

print('Running multi-turn conversation on animal scene...')
history_animal = multiturn(img_animal, questions_animal)
show_demo(img_animal, history_animal, title='Animal Scene')

## Demo 3 — Urban / Street Scene (Multi-Turn)

In [ ]:
img_urban = fetch_image('http://images.cocodataset.org/train2017/000000000030.jpg')

questions_urban = [
    'What is the main subject of this image?',
    'What vehicles or objects can you identify?',
    'What is happening in the background?',
    'What city or location does this look like?',
]

print('Running multi-turn conversation on urban scene...')
history_urban = multiturn(img_urban, questions_urban)
show_demo(img_urban, history_urban, title='Urban Scene')

## Demo 4 — Sports / Action (Multi-Turn)

In [ ]:
img_sports = fetch_image('http://images.cocodataset.org/train2017/000000000034.jpg')

questions_sports = [
    'What sport or activity is shown in this image?',
    'How many people are visible?',
    'What are the people wearing?',
    'Who appears to be winning or performing better?',
]

print('Running multi-turn conversation on sports scene...')
history_sports = multiturn(img_sports, questions_sports)
show_demo(img_sports, history_sports, title='Sports Scene')

## Demo 5 — Before vs After Fine-tuning

We disable LoRA adapters to simulate the base (non-fine-tuned) model, then re-enable them.  
This shows what fine-tuning actually added.

In [ ]:
test_image = img_food  # reuse food image

test_questions = [
    'What food items are visible in this image?',
    'Is this a balanced meal?',
]

# ── Fine-tuned (LoRA ON) ──────────────────────────────────────────────────────
model.enable_adapter_layers()
finetuned_history = multiturn(test_image, test_questions)

# ── Base model (LoRA OFF) ─────────────────────────────────────────────────────
model.disable_adapter_layers()
base_history = multiturn(test_image, test_questions)
model.enable_adapter_layers()  # re-enable for subsequent demos

# ── Plot comparison ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5),
                          gridspec_kw={'width_ratios': [1, 1.8, 1.8]})

axes[0].imshow(test_image)
axes[0].axis('off')
axes[0].set_title('Test Image', fontsize=11, fontweight='bold')

for ax, history, label, color in [
    (axes[1], base_history,      'Base PaliGemma-3B\n(No fine-tuning)', '#dc2626'),
    (axes[2], finetuned_history, 'Fine-tuned PaliGemma-3B\n(LoRA on LLaVA-Instruct)', '#16a34a'),
]:
    ax.axis('off')
    ax.set_title(label, fontsize=10, fontweight='bold', color=color)
    y = 0.95
    for i, turn in enumerate(history):
        ax.text(0.02, y, f'Q{i+1}: {turn["q"]}',
                transform=ax.transAxes, fontsize=9,
                color='#374151', fontweight='bold', va='top')
        y -= 0.1
        ax.text(0.02, y, f'A{i+1}: {turn["a"]}',
                transform=ax.transAxes, fontsize=9,
                color='#1f2937', va='top',
                bbox=dict(boxstyle='round,pad=0.4',
                          facecolor='#dcfce7' if color == '#16a34a' else '#fee2e2',
                          alpha=0.6))
        y -= 0.35

plt.suptitle('Before vs After Fine-tuning', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/demo_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

## Demo 6 — Ablation: Does History Help?

Test the same final question with 0, 1, 2, and 3 turns of prior context.

In [ ]:
ablation_image = img_food

# Full conversation we'll slice from
full_conv = [
    {'q': 'What food items are in this image?',
     'a': 'Fruits, vegetables, nuts, bread, cookies, and meat.'},
    {'q': 'What colors do the containers have?',
     'a': 'The containers are pink, yellow, and blue.'},
    {'q': 'Are the foods arranged neatly?',
     'a': 'Yes, the foods are arranged in separate compartments neatly.'},
]
final_q = 'Based on everything you see, would you say this is a meal for one person or multiple people?'

results = {}
for n_turns in [0, 1, 2, 3]:
    history = full_conv[:n_turns]
    parts   = [f'Q: {h["q"]} A: {h["a"]}' for h in history]
    parts.append(f'Q: {final_q} A:')
    answer  = generate(ablation_image, ' '.join(parts))
    results[n_turns] = answer
    print(f'History {n_turns} turn(s): {answer[:120]}')

# Visualise
fig, axes = plt.subplots(1, 5, figsize=(18, 4),
                          gridspec_kw={'width_ratios': [1, 1.2, 1.2, 1.2, 1.2]})
axes[0].imshow(ablation_image)
axes[0].axis('off')
axes[0].set_title('Image', fontweight='bold')
axes[0].set_xlabel(f'Q: {final_q[:50]}…', fontsize=8)

colors = ['#fecdd3', '#fde68a', '#bbf7d0', '#bfdbfe']
for i, (n, answer) in enumerate(results.items()):
    ax = axes[i + 1]
    ax.axis('off')
    ax.set_title(f'{n} Prior Turn{"s" if n != 1 else ""}', fontsize=10, fontweight='bold')
    ax.text(0.5, 0.5, answer[:200], transform=ax.transAxes,
            fontsize=8.5, va='center', ha='center', wrap=True,
            bbox=dict(boxstyle='round,pad=0.5', facecolor=colors[i], alpha=0.8),
            multialignment='center')

plt.suptitle('Ablation: Conversation History Length vs Answer Quality',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/demo_ablation.png', dpi=130, bbox_inches='tight')
plt.show()

## Summary Dashboard

In [ ]:
import os

fig = plt.figure(figsize=(14, 9))
fig.patch.set_facecolor('#1e1b4b')

# Title
fig.text(0.5, 0.95, 'PaliGemma-3B LoRA — DLCV Project Results',
         ha='center', va='top', fontsize=16, fontweight='bold', color='white')
fig.text(0.5, 0.91, 'Fine-tuned on LLaVA-Instruct-150K · Kaggle T4 GPU · 1h 15min training',
         ha='center', va='top', fontsize=10, color='#a5b4fc')

# Metrics
metrics = [
    ('3.2B', 'Total\nParameters'),
    ('18.1M', 'Trainable\n(LoRA)'),
    ('0.56%', 'Params\nTrained'),
    ('2.4→1.1', 'Loss\nReduction'),
    ('1,800', 'Training\nSamples'),
    ('1h 15m', 'Training\nTime'),
]

colors_m = ['#6366f1','#8b5cf6','#a78bfa','#059669','#d97706','#dc2626']
for i, (val, lbl) in enumerate(metrics):
    x = 0.08 + i * 0.155
    ax = fig.add_axes([x, 0.72, 0.13, 0.13])
    ax.set_facecolor(colors_m[i])
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.6, val, ha='center', va='center',
            fontsize=14, fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.15, lbl, ha='center', va='center',
            fontsize=8, color='rgba(255,255,255,0.8)', transform=ax.transAxes)

# Training curves (load saved image from training)
curves_path = '/kaggle/working/dlcv_paligemma/training_curves.png'
if not os.path.exists(curves_path):
    curves_paths = glob.glob('/kaggle/input/**/training_curves.png', recursive=True)
    curves_path  = curves_paths[0] if curves_paths else None

if curves_path and os.path.exists(curves_path):
    ax_curves = fig.add_axes([0.05, 0.35, 0.42, 0.30])
    ax_curves.imshow(plt.imread(curves_path))
    ax_curves.axis('off')
    ax_curves.set_title('Training & Validation Loss', color='white', fontsize=10, pad=4)

# Architecture text box
ax_arch = fig.add_axes([0.52, 0.35, 0.44, 0.30])
ax_arch.set_facecolor('#312e81')
for spine in ax_arch.spines.values(): spine.set_color('#6366f1')
ax_arch.set_xticks([]); ax_arch.set_yticks([])
arch_text = (
    'Model Architecture\n'
    '─────────────────────────\n'
    '🖼  SigLIP ViT-So/14      FROZEN\n'
    '    1024 visual tokens\n\n'
    '⟶  Linear Projector       FROZEN\n'
    '    Image → Language space\n\n'
    '💬  Gemma-2B LLM          FROZEN\n'
    '    + LoRA Adapters       TRAINED\n'
    '    r=8, α=16, 7 modules\n'
    '    18.1M trainable params\n\n'
    '⟶  Multi-Turn Answer'
)
ax_arch.text(0.05, 0.95, arch_text, transform=ax_arch.transAxes,
             fontsize=9, va='top', color='#e0e7ff',
             fontfamily='monospace')

# Demo sample box
ax_demo = fig.add_axes([0.05, 0.04, 0.90, 0.26])
ax_demo.set_facecolor('#1e3a5f')
for spine in ax_demo.spines.values(): spine.set_color('#3b82f6')
ax_demo.set_xticks([]); ax_demo.set_yticks([])

if history_food:
    demo_lines = ['Multi-Turn Demo (Food Scene) — Fine-tuned PaliGemma-3B', '']
    for i, h in enumerate(history_food[:3]):
        demo_lines.append(f'Q{i+1}: {h["q"]}')
        demo_lines.append(f'A{i+1}: {h["a"][:100]}{"..." if len(h["a"]) > 100 else ""}')
        demo_lines.append('')
    ax_demo.text(0.02, 0.95, '\n'.join(demo_lines), transform=ax_demo.transAxes,
                 fontsize=8.5, va='top', color='#e0e7ff', fontfamily='monospace')

plt.savefig('/kaggle/working/demo_summary_dashboard.png', dpi=140,
            bbox_inches='tight', facecolor='#1e1b4b')
plt.show()
print('\n✅ All demo outputs saved to /kaggle/working/')
print('\nFiles saved:')
for f in sorted(os.listdir('/kaggle/working/')):
    if f.startswith('demo_'):
        size = os.path.getsize(f'/kaggle/working/{f}') / 1024
        print(f'  {f}  ({size:.0f} KB)')